In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)

FORECAST_HORIZON = 30
TEST_DAYS = 60
OPTUNA_TRIALS = 120


In [2]:
### Charger
df = pd.read_csv("sales_step4_b2c.csv", encoding="utf-8-sig")
df["sale_date"] = pd.to_datetime(df["sale_date"])
df = df.sort_values(["product_id", "sale_date"]).reset_index(drop=True)
products = df["product_id"].unique()
print(f" Dataset B2C : {len(df):,} lignes, {len(products)} produits")

 Dataset B2C : 275,867 lignes, 1287 produits


In [3]:
### features
print("\n  Création des features...")
df["days_since_last_sale"] = 0
for i, pid in enumerate(products):
    if (i + 1) % 300 == 0:
        print(f"    {i+1}/{len(products)}...")
    mask = df["product_id"] == pid
    product_df = df.loc[mask]
    sale_dates = product_df.loc[product_df["qty_sold"] > 0, "sale_date"]
    if len(sale_dates) == 0:
        df.loc[mask, "days_since_last_sale"] = 999
        continue
    for idx in product_df.index:
        current_date = df.loc[idx, "sale_date"]
        past_sales = sale_dates[sale_dates < current_date]
        if len(past_sales) > 0:
            df.loc[idx, "days_since_last_sale"] = (current_date - past_sales.max()).days
        else:
            df.loc[idx, "days_since_last_sale"] = 999

df["sale_frequency_30d"] = df.groupby("product_id")["qty_sold"].transform(
    lambda x: (x.shift(1) > 0).rolling(window=30, min_periods=1).sum()
)
df["max_sale_30d"] = df.groupby("product_id")["qty_sold"].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=1).max()
)
df["total_sales_7d"] = df.groupby("product_id")["qty_sold"].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).sum()
)
df["total_sales_30d"] = df.groupby("product_id")["qty_sold"].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=1).sum()
)
avg_price = df[df["prix_unitaire"] > 0].groupby("product_id")["prix_unitaire"].mean()
df["avg_price"] = df["product_id"].map(avg_price).fillna(0)
first_sale = df[df["qty_sold"] > 0].groupby("product_id")["sale_date"].min()
df["first_sale_date"] = df["product_id"].map(first_sale)
df["product_age_days"] = (df["sale_date"] - df["first_sale_date"]).dt.days
df["product_age_days"] = df["product_age_days"].fillna(0).clip(lower=0)
df = df.drop(columns=["first_sale_date"])
df = df.replace([np.inf, -np.inf], 0).fillna(0)
print("  ✓ Features de base créées")


  Création des features...
    300/1287...
    600/1287...
    900/1287...
    1200/1287...
  ✓ Features de base créées


In [4]:
# 3. Agrégation
print(f"\n  Agrégation {FORECAST_HORIZON}j...")
cutoff_date = df["sale_date"].max() - pd.Timedelta(days=TEST_DAYS)
windows_data = []

for i, pid in enumerate(products):
    if (i + 1) % 300 == 0:
        print(f"    {i+1}/{len(products)}...")
    product_df = df[df["product_id"] == pid]
    if len(product_df) < FORECAST_HORIZON + 7:
        continue
    for j in range(0, len(product_df) - FORECAST_HORIZON, 7):
        cr = product_df.iloc[j]
        cd = cr["sale_date"]
        fm = (product_df["sale_date"] > cd) & \
             (product_df["sale_date"] <= cd + pd.Timedelta(days=FORECAST_HORIZON))
        fs = product_df.loc[fm, "qty_sold"].sum()
        windows_data.append({
            "product_id": pid, "category_id": cr["category_id"],
            "sale_date": cd, "month": cr["month"], "quarter": cr["quarter"],
            "day_of_week": cr["day_of_week"], "is_weekend": cr["is_weekend"],
            "is_christmas_period": cr["is_christmas_period"],
            "is_summer": cr["is_summer"], "is_back_to_school": cr["is_back_to_school"],
            "is_black_friday_period": cr["is_black_friday_period"],
            "rolling_mean_7": cr["rolling_mean_7"], "rolling_mean_14": cr["rolling_mean_14"],
            "rolling_mean_30": cr["rolling_mean_30"], "rolling_mean_60": cr["rolling_mean_60"],
            "rolling_mean_90": cr["rolling_mean_90"],
            "rolling_std_7": cr["rolling_std_7"], "rolling_std_30": cr["rolling_std_30"],
            "trend_7_30": cr["trend_7_30"], "trend_30_90": cr["trend_30_90"],
            "lag_7": cr["lag_7"], "lag_14": cr["lag_14"], "lag_28": cr["lag_28"],
            "days_since_last_sale": cr["days_since_last_sale"],
            "sale_frequency_30d": cr["sale_frequency_30d"],
            "max_sale_30d": cr["max_sale_30d"],
            "total_sales_7d": cr["total_sales_7d"],
            "total_sales_30d": cr["total_sales_30d"],
            "avg_price": cr["avg_price"], "product_age_days": cr["product_age_days"],
            "target_30d": fs,
        })

df_agg = pd.DataFrame(windows_data)
df_agg["sale_date"] = pd.to_datetime(df_agg["sale_date"])
print(f" Agrégé : {len(df_agg):,} lignes")


  Agrégation 30j...
    300/1287...
    600/1287...
    900/1287...
    1200/1287...
 Agrégé : 34,572 lignes


In [5]:
# ════════════════════════════════════════
# 4. AMÉLIORATIONS
# ════════════════════════════════════════
print("\n  Améliorations...")

# 4a. Retirer les zéros du dataset agrégé
avant = len(df_agg)
df_agg = df_agg[df_agg["target_30d"] > 0].copy()
print(f"  ✓ Retrait zéros : {avant:,} → {len(df_agg):,} lignes")

# 4b. Features d'interaction
df_agg["volume_x_freq"] = df_agg["rolling_mean_30"] * df_agg["sale_frequency_30d"]
df_agg["recent_vs_old"] = df_agg["total_sales_7d"] / df_agg["total_sales_30d"].replace(0, 1)
df_agg["demand_stability"] = df_agg["rolling_std_30"] / df_agg["rolling_mean_30"].replace(0, 1)
print("  ✓ Features interaction : volume_x_freq, recent_vs_old, demand_stability")

# 4c. Log-transform target
df_agg["target_log"] = np.log1p(df_agg["target_30d"])
print(f"  ✓ Log-transform target")

# ════════════════════════════════════════
# 5. SPLIT + FEATURES RÉDUITES
# ════════════════════════════════════════
# Retirer les features redondantes (corrélation > 0.9 entre elles)
features_to_remove = ["total_sales_7d", "total_sales_30d", "rolling_mean_90", "quarter"]

feature_cols = [c for c in df_agg.columns if c not in ["sale_date", "target_30d", "target_log", "product_id"]]
feature_cols = [c for c in feature_cols if c not in features_to_remove]

train_agg = df_agg[df_agg["sale_date"] <= cutoff_date]
test_agg = df_agg[df_agg["sale_date"] > cutoff_date]
X_train = train_agg[feature_cols]
y_train_log = train_agg["target_log"]
y_train_real = train_agg["target_30d"]
X_test = test_agg[feature_cols]
y_test = test_agg["target_30d"]

train_weights = 1 + np.log1p(y_train_real)

print(f"\n  Train : {len(train_agg):,} | Test : {len(test_agg):,}")
print(f"  Features : {len(feature_cols)} (retiré {features_to_remove})")

# ════════════════════════════════════════
# 6. OPTUNA AVEC CV INTÉGRÉE
# ════════════════════════════════════════
print(f"\n  Tuning Optuna ({OPTUNA_TRIALS} essais) avec CV intégrée...")

df_agg_sorted = df_agg.sort_values("sale_date").reset_index(drop=True)
X_all = df_agg_sorted[feature_cols]
y_all_log = np.log1p(df_agg_sorted["target_30d"])
y_all_real = df_agg_sorted["target_30d"]

# TimeSeriesSplit configuré : test = 20% par fold
test_size = int(len(X_all) * 0.20)
tscv = TimeSeriesSplit(n_splits=4, test_size=test_size)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5.0),
        "random_state": 42, "n_jobs": -1,
    }
    
    mae_scores = []
    for train_idx, test_idx in tscv.split(X_all):
        X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
        y_tr = y_all_log.iloc[train_idx]
        y_te_real = y_all_real.iloc[test_idx]
        w = 1 + np.log1p(y_all_real.iloc[train_idx])
        
        m = xgb.XGBRegressor(**params)
        m.fit(X_tr, y_tr, sample_weight=w, verbose=False)
        preds_log = m.predict(X_te)
        preds = np.maximum(0, np.expm1(preds_log))
        mae_scores.append(mean_absolute_error(y_te_real, preds))
    
    return np.mean(mae_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=OPTUNA_TRIALS)
best_params = study.best_params
print(f"  ✓ Best CV MAE : {study.best_value:.3f}")


  Améliorations...
  ✓ Retrait zéros : 34,572 → 29,286 lignes
  ✓ Features interaction : volume_x_freq, recent_vs_old, demand_stability
  ✓ Log-transform target

  Train : 29,056 | Test : 230
  Features : 27 (retiré ['total_sales_7d', 'total_sales_30d', 'rolling_mean_90', 'quarter'])

  Tuning Optuna (120 essais) avec CV intégrée...
  ✓ Best CV MAE : 2.350


In [6]:
# 7. Entraînement final (sur log target)
print(f"\n  Entraînement final...")
model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
model.fit(X_train, y_train_log, sample_weight=train_weights, verbose=False)


  Entraînement final...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6283509938678403, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=1.0158450676334532, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.015070504766144395,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=9, max_leaves=None,
             min_child_weight=4, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=601, n_jobs=-1,
             num_parallel_tree=None, ...)

In [7]:
# Prédictions (inverse log)
y_pred = np.maximum(0, np.expm1(model.predict(X_test)))
y_pred_rounded = np.round(y_pred).astype(int)

# Train predictions
y_pred_train = np.maximum(0, np.expm1(model.predict(X_train)))

In [15]:
# 8. Métriques
from sklearn.metrics import median_absolute_error

mae = mean_absolute_error(y_test, y_pred)
mae_train = mean_absolute_error(y_train_real, y_pred_train)
wmape = np.sum(np.abs(y_test.values - y_pred)) / np.sum(y_test.values) * 100
wmape_train = np.sum(np.abs(y_train_real.values - y_pred_train)) / np.sum(y_train_real.values) * 100
errors = np.abs(y_test.values - y_pred)
acc_3 = (errors <= 3).mean() * 100

print("\n" + "=" * 60)
print("MÉTRIQUES B2C")
print("=" * 60)

print(f"\n  ── Overfitting check ──")
print(f"  {'Métrique':<12s} {'Train':>10s} {'Test':>10s} {'Écart':>10s}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'MAE':<12s} {mae_train:>10.3f} {mae:>10.3f} {mae-mae_train:>+10.3f}")
print(f"  {'WMAPE':<12s} {wmape_train:>9.1f}% {wmape:>9.1f}% {wmape-wmape_train:>+9.1f}%")
print(f"\n  ── Métriques test ──")
print(f"  MAE       : {mae:.3f} unités/30j")
print(f"  WMAPE     : {wmape:.1f}%")
print(f"  Acc ±3    : {acc_3:.1f}%")


MÉTRIQUES B2C

  ── Overfitting check ──
  Métrique          Train       Test      Écart
  ------------ ---------- ---------- ----------
  MAE               1.475      2.541     +1.067
  WMAPE             35.5%      43.9%      +8.4%

  ── Métriques test ──
  MAE       : 2.541 unités/30j
  WMAPE     : 43.9%
  Acc ±3    : 76.1%


In [16]:
# 9. Cross-validation finale
from sklearn.metrics import median_absolute_error

print("\n" + "=" * 50)
print("CROSS-VALIDATION FINALE")
print("=" * 50)

cv_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_tr = y_all_log.iloc[train_idx]
    y_te_real = y_all_real.iloc[test_idx]
    w = 1 + np.log1p(y_all_real.iloc[train_idx])

    train_pct = len(train_idx) / len(X_all) * 100
    test_pct = len(test_idx) / len(X_all) * 100
    
    fold_model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    fold_model.fit(X_tr, y_tr, sample_weight=w, verbose=False)
    preds = np.maximum(0, np.expm1(fold_model.predict(X_te)))
    
    fold_mae = mean_absolute_error(y_te_real, preds)
    fold_wmape = np.sum(np.abs(y_te_real.values - preds)) / np.sum(y_te_real.values) * 100
    fold_errors = np.abs(y_te_real.values - preds)
    fold_acc3 = (fold_errors <= 3).mean() * 100
    
    cv_results.append({
        "fold": fold+1, "mae": fold_mae, 
        "wmape": fold_wmape, "acc_3": fold_acc3,
    })
    print(f"\n  Fold {fold+1} : train={len(train_idx):,} ({train_pct:.0f}%), test={len(test_idx):,} ({test_pct:.0f}%)")
    print(f"    MAE={fold_mae:.3f}, WMAPE={fold_wmape:.1f}%, Acc±3={fold_acc3:.1f}%")

cv_df = pd.DataFrame(cv_results)
print(f"\n  {'='*40}")
print(f"  RÉSULTAT FINAL")
print(f"  {'='*40}")
print(f"  MAE     : {cv_df['mae'].mean():.3f} ± {cv_df['mae'].std():.3f}")
print(f"  WMAPE   : {cv_df['wmape'].mean():.1f}% ± {cv_df['wmape'].std():.1f}%")
print(f"  Acc ±3  : {cv_df['acc_3'].mean():.1f}% ± {cv_df['acc_3'].std():.1f}%")
print(f"  Features: {len(feature_cols)}")


CROSS-VALIDATION FINALE

  Fold 1 : train=5,858 (20%), test=5,857 (20%)
    MAE=2.364, WMAPE=57.3%, Acc±3=78.0%

  Fold 2 : train=11,715 (40%), test=5,857 (20%)
    MAE=1.925, WMAPE=52.5%, Acc±3=83.8%

  Fold 3 : train=17,572 (60%), test=5,857 (20%)
    MAE=2.692, WMAPE=56.6%, Acc±3=78.2%

  Fold 4 : train=23,429 (80%), test=5,857 (20%)
    MAE=2.419, WMAPE=53.3%, Acc±3=76.7%

  RÉSULTAT FINAL
  MAE     : 2.350 ± 0.318
  WMAPE   : 54.9% ± 2.4%
  Acc ±3  : 79.2% ± 3.2%
  Features: 27


In [17]:
print(f"\n  {'='*40}")
print(f"  RÉSULTAT FINAL")
print(f"  {'='*40}")
print(f"  MAE  : {cv_df['mae'].mean():.3f} ± {cv_df['mae'].std():.3f}")
#print(f"  R²   : {cv_df['r2'].mean():.4f} ± {cv_df['r2'].std():.4f}")
#print(f"  MAPE : {cv_df['mape'].mean():.1f}% ± {cv_df['mape'].std():.1f}%")

print(f"\n  ── Évolution ──")

#print(f"  B2C amélioré    : R²={cv_df['r2'].mean():.4f} ± {cv_df['r2'].std():.4f}, MAE={cv_df['mae'].mean():.3f}")
print(f"  B2C amélioré    :  MAE={cv_df['mae'].mean():.3f}")




  RÉSULTAT FINAL
  MAE  : 2.350 ± 0.318

  ── Évolution ──
  B2C amélioré    :  MAE=2.350


In [18]:
print("\n" + "-" * 40)
print("PRÉDICTIONS SUR 30 JOURS — TOP 20 PRODUITS")
print("-" * 40)
 
test_agg_result = test_agg.copy()
test_agg_result["predicted_30d"] = y_pred_rounded
 
product_preds = (
    test_agg_result.sort_values("sale_date")
    .groupby("product_id").last().reset_index()
)
product_names = df.groupby("product_id")["product_name"].first().reset_index()
product_preds = product_preds.merge(product_names, on="product_id", how="left")
 
top20 = product_preds.nlargest(20, "target_30d")
 
print(f"\n  {'Produit':<45s} {'Réel':>6s} {'Prédit':>6s} {'Écart':>7s}")
print(f"  {'-'*45} {'-'*6} {'-'*6} {'-'*7}")
for _, row in top20.iterrows():
    name = str(row["product_name"])[:45]
    ecart = row["predicted_30d"] - row["target_30d"]
    indicator = "✓" if abs(ecart) <= 3 else ("~" if abs(ecart) <= 10 else "✗")
    print(f"  {name:<45s} {int(row['target_30d']):>6d} {int(row['predicted_30d']):>6d} {ecart:>+6.0f} {indicator}")
 
# Compter les bonnes prédictions
all_preds = product_preds[product_preds["target_30d"] > 0]
n_good = ((all_preds["predicted_30d"] - all_preds["target_30d"]).abs() <= 3).sum()
n_ok = ((all_preds["predicted_30d"] - all_preds["target_30d"]).abs() <= 10).sum()
n_total = len(all_preds)
print(f"\n  Sur tous les produits avec ventes :")
print(f"    ✓ Écart ≤ 3  : {n_good}/{n_total} ({n_good/n_total*100:.0f}%)")
print(f"    ~ Écart ≤ 10 : {n_ok}/{n_total} ({n_ok/n_total*100:.0f}%)")


----------------------------------------
PRÉDICTIONS SUR 30 JOURS — TOP 20 PRODUITS
----------------------------------------

  Produit                                         Réel Prédit   Écart
  --------------------------------------------- ------ ------ -------
  Livecards Playsation 25€                          22      8    -14 ✗
  Lenovo Thinkpad T490s                             22      7    -15 ✗
  Epson EcoTank L3250                               19     10     -9 ~
  Livecards PS Plus 50€                             17     10     -7 ~
  Microphone JBL Wireless Noir                      15     10     -5 ~
  JBL PARTYBOX STAGE 320                            13      5     -8 ~
  PLUGGER Cable Tweed Jack Male Mono - Jack Mal     12      5     -7 ~
  Konix Mythics Câble USB C PS5 3m                   9      3     -6 ~
  Onduleur UPS Technology 1100VA In Line             9      7     -2 ✓
  Urban Factory Urban Sleeve 14"                     7      5     -2 ✓
  Epson 103 Ecotank Bla

In [ ]:
#### CROSS VALIDATION SLICING

In [13]:
# ══════════════════════════════════════════════
# MATRICE DE CORRÉLATION DES FEATURES
# ══════════════════════════════════════════════
import plotly.figure_factory as ff

# Calculer la corrélation sur le dataset agrégé (sans les zéros)
corr_cols = feature_cols + ["target_30d"]
corr_matrix = df_agg[corr_cols].corr()

# Plotly heatmap
fig = ff.create_annotated_heatmap(
    z=corr_matrix.values.round(2),
    x=corr_cols,
    y=corr_cols,
    colorscale="RdBu_r",
    showscale=True,
    reversescale=True,
)

fig.update_layout(
    title="Matrice de corrélation des features",
    width=1200,
    height=1000,
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
)

fig.write_html("correlation_matrix.html")
fig.show()

# Corrélation avec le target (trié)
print("\n  Corrélation avec target_30d :")
target_corr = corr_matrix["target_30d"].drop("target_30d").sort_values(ascending=False)
for feat, corr in target_corr.items():
    bar = "█" * int(abs(corr) * 40)
    sign = "+" if corr > 0 else "-"
    print(f"    {feat:25s} {sign}{abs(corr):.3f}  {bar}")

# Détecter les features très corrélées entre elles (>0.9)
print(f"\n  Features fortement corrélées entre elles (>0.9) :")
import itertools
high_corr = []
for f1, f2 in itertools.combinations(feature_cols, 2):
    c = abs(corr_matrix.loc[f1, f2])
    if c > 0.9:
        high_corr.append((f1, f2, c))
        print(f"    {f1:25s} ↔ {f2:25s} : {c:.3f}")

if not high_corr:
    print(f"    Aucune — pas de multicolinéarité excessive")
else:
    print(f"\n    → {len(high_corr)} paires fortement corrélées.")
    print(f"    Pour XGBoost ce n'est PAS un problème (contrairement à la régression linéaire).")
    print(f"    L'arbre choisit simplement la meilleure des deux, il n'est pas perturbé.")


  Corrélation avec target_30d :
    rolling_mean_30           +0.508  ████████████████████
    rolling_mean_60           +0.485  ███████████████████
    volume_x_freq             +0.466  ██████████████████
    rolling_mean_14           +0.464  ██████████████████
    sale_frequency_30d        +0.451  ██████████████████
    rolling_mean_7            +0.385  ███████████████
    rolling_std_30            +0.378  ███████████████
    max_sale_30d              +0.276  ███████████
    rolling_std_7             +0.249  █████████
    lag_14                    +0.204  ████████
    lag_7                     +0.177  ███████
    lag_28                    +0.163  ██████
    month                     +0.086  ███
    is_black_friday_period    +0.077  ███
    product_age_days          +0.041  █
    trend_30_90               +0.037  █
    recent_vs_old             +0.030  █
    is_christmas_period       +0.008  
    is_back_to_school         -0.007  
    is_weekend                -0.008  
    day_of_wee

### Enregistre dans MLFLow

In [ ]:
import mlflow
import mlflow.xgboost
import os

mlflow.set_tracking_uri("https://stephane-nxt-demodaymlflow.hf.space/")
mlflow.set_experiment("experiment-smart-reassort-xgboost-forecast-30d-v5")

with mlflow.start_run(run_name="smart-reassort-30d-v5"):
    # Paramètres
    mlflow.log_params(best_params)
    mlflow.log_param("forecast_horizon", FORECAST_HORIZON)
    mlflow.log_param("test_days", TEST_DAYS)
    mlflow.log_param("optuna_trials", OPTUNA_TRIALS)
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("features_removed", "total_sales_7d, total_sales_30d, rolling_mean_90, quarter")
    mlflow.log_param("n_products", df["product_id"].nunique())
    mlflow.log_param("sample_weight", "1+log1p")
    mlflow.log_param("target_transform", "log1p")
    mlflow.log_param("cv_method", "TimeSeriesSplit_4fold_20pct")
    mlflow.log_param("features_list", ", ".join(feature_cols))

    # Métriques split unique
    mlflow.log_metric("mae_test", mae)
    mlflow.log_metric("wmape_test", wmape)
    mlflow.log_metric("acc3_test", acc_3)
    mlflow.log_metric("mae_train", mae_train)
    mlflow.log_metric("wmape_train", wmape_train)

    # Métriques CV
    mlflow.log_metric("cv_mae_mean", cv_df["mae"].mean())
    mlflow.log_metric("cv_mae_std", cv_df["mae"].std())
    mlflow.log_metric("cv_wmape_mean", cv_df["wmape"].mean())
    mlflow.log_metric("cv_wmape_std", cv_df["wmape"].std())
    mlflow.log_metric("cv_acc3_mean", cv_df["acc_3"].mean())
    mlflow.log_metric("cv_acc3_std", cv_df["acc_3"].std())

    run_id = mlflow.active_run().info.run_id



2026/03/19 09:53:45 INFO mlflow.tracking.fluent: Experiment with name 'experiment-smart-reassort-xgboost-forecast-30d-v5' does not exist. Creating a new experiment.


🏃 View run smart-reassort-30d-v5 at: https://stephane-nxt-demodaymlflow.hf.space/#/experiments/6/runs/fcfd3165227542d7bb30be239f8c3a5b
🧪 View experiment at: https://stephane-nxt-demodaymlflow.hf.space/#/experiments/6


In [25]:
# Sauvegarder le modèle en pickle (en local, pas sur S3)
import pickle
with open("xgboost_30d_final.pkl", "wb") as f:
    pickle.dump({
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "forecast_horizon": FORECAST_HORIZON,
    }, f)

print(f"✓ MLflow enregistré (params + métriques)")
print(f"  Run ID : {run_id}")
print(f"✓ Modèle sauvé en local : xgboost_30d_final.pkl")
print(f"\n  Note : le modèle n'est pas sur MLflow (erreur S3)")
print(f"  → Configurer les credentials S3 plus tard pour log_model")

✓ MLflow enregistré (params + métriques)
  Run ID : fcfd3165227542d7bb30be239f8c3a5b
✓ Modèle sauvé en local : xgboost_30d_final.pkl

  Note : le modèle n'est pas sur MLflow (erreur S3)
  → Configurer les credentials S3 plus tard pour log_model
